<a href="https://colab.research.google.com/github/jaswanth82006/firstml/blob/main/bus_api_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask flask-cors pyngrok joblib pandas scikit-learn

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import pandas as pd

# Load model & encoders
model = joblib.load("/content/bus_model.pkl")
le_route = joblib.load("/content/route_encoder.pkl")
le_stop = joblib.load("/content/stop_encoder.pkl")
le_weather = joblib.load("/content/weather_encoder.pkl")

FEATURE_ORDER = ['hour', 'route', 'stop', 'weather', 'weekday']
TOTAL_SEATS = 30

app = Flask(__name__)
CORS(app)

@app.route("/")
def home():
    return "Bus Occupancy Prediction API (Colab) is running"

# GET method
@app.route("/predict", methods=["GET"])
def predict():
    try:
        route = request.args.get("route")
        stop = request.args.get("stop")
        hour = int(request.args.get("hour"))
        weekday = int(request.args.get("weekday"))
        weather = request.args.get("weather")

        route_enc = le_route.transform([route])[0]
        stop_enc = le_stop.transform([stop])[0]
        weather_enc = le_weather.transform([weather])[0]

        sample = pd.DataFrame([[
            hour,
            route_enc,
            stop_enc,
            weather_enc,
            weekday
        ]], columns=FEATURE_ORDER)

        predicted_passengers = int(model.predict(sample)[0])
        overcrowded = predicted_passengers > TOTAL_SEATS

        reasons = []

        if 16 <= hour <= 19:
            reasons.append("peak office hour")

        reasons.append("working day" if weekday == 1 else "weekend")
        reasons.append("rainy weather" if weather == "rain" else "normal weather")

        return jsonify({
            "predicted_passengers": predicted_passengers,
            "total_seats": TOTAL_SEATS,
            "status": "Overcrowded" if overcrowded else "Not Overcrowded",
            "explanation": "Because of " + ", ".join(reasons)
        })

    except Exception as e:
        return jsonify({"error": str(e)})


In [ ]:

from pyngrok import ngrok

ngrok.set_auth_token("39DXONJ2qvizAZDLO83VygmxL76_2v8yNaZPswVkHzA9kqpiG")

In [ ]:
import threading

def run_flask():
    app.run(
        port=5000,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(target=run_flask)
thread.start()

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print("✅ Public URL:", public_url)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Public URL: NgrokTunnel: "https://ozonous-vivien-pancratic.ngrok-free.dev" -> "http://localhost:5000"
